In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client

In [2]:
client = Client()
client = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/StoichModelGP/ModelGP_M05.json")
client.get_next_trials(max_trials=1)

{56: {'n_ci': 0.7798925806879506, 'n_it': 0.3750300299425563}}

In [3]:
def PredictorsToCaStoichs(s1,b1):
    return (0.5+(2.0-0.5)*s1)*b1

def PredictorsToIaStoichs(s2,b1):
    return (1.0+(2.0-1.0)*s2)*(1-b1)

In [4]:
trials = 300

X_lis = []
for i in range(trials):
    sampler = Sampler_class()
    Parameters_lis = [
        {"name":"x1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x3", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    for i in range(100):
        X = sampler.three.PseudorandomSampler3D_func(27,Parameters_lis).T
        if np.shape(X) != (0,3):
            break
    X_lis.append(X)

y_max_lis = []
for X in X_lis:
    s1_arr = X.T[0]
    s2_arr = X.T[1]
    b1_arr = X.T[2]
    n_ci_lis = []
    for s1,b1 in zip(s1_arr,b1_arr):
        n_ci_lis.append(PredictorsToCaStoichs(s1,b1))
    n_it_lis = []
    for s2,b1 in zip(s2_arr,b1_arr):
        n_it_lis.append(PredictorsToIaStoichs(s2,b1))
    n_ci_arr = np.array(n_ci_lis)
    n_it_arr = np.array(n_it_lis)
    y_pred_lis = []
    for n_ci,n_it in zip(n_ci_arr,n_it_arr):
        y_pred_lis.append(client.predict([{"n_ci":n_ci,"n_it":n_it}])[0]["t1"][0])
    y_max_lis.append(np.max(np.array(y_pred_lis)))

y_max_arr = np.array(y_max_lis)
print(y_max_arr.tolist())
print(np.average(y_max_arr))

[13.96157936311837, 13.616380798471727, 13.709016969938865, 13.598574959457606, 16.950663045655585, 14.033694874264096, 15.504206772688828, 17.607220943191535, 15.045833390573973, 14.002136168094815, 16.047675697857215, 15.280592352531686, 14.772550056419108, 13.588075479149682, 14.361845779982868, 14.824536561199052, 15.463338852538419, 15.576875467022097, 13.754936083706616, 14.522777249755345, 14.221038891181802, 14.593312712322046, 13.786150248476153, 14.397224916551025, 13.66531377244111, 15.062370769723003, 15.948438379850947, 14.006979705020816, 16.959438655832102, 15.10137236247024, 16.30544218912394, 15.451620471926773, 14.995168317278946, 16.914293282828115, 13.965980185538559, 13.642498649053106, 15.979785693193044, 15.955683449847301, 14.142154808382013, 15.182724110593039, 13.96671438071561, 16.004412161577687, 15.213039167435308, 13.567223974702223, 13.581729885845482, 13.674465547722228, 16.380975072755593, 14.440941268556527, 13.664640193409161, 15.938065154860436, 14.2

In [5]:
np.average(y_max_arr)

np.float64(14.703186422040401)

In [6]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/TestPredModelGP_M05/DataGenerated/pseudorandom_27.pkl"
latestdf = pd.DataFrame(y_max_arr)
pd.to_pickle(obj=latestdf,filepath_or_buffer=filepath)